# Byte Pair Encoding (BPE), Data Loaders & Input Embeddings
## Chapter 2: Preparing Input Tensors for Large Language Models

In this notebook, we advance beyond naive character/word-level tokenizers by implementing production-grade input preparation pipelines used in models like **GPT-2** and **GPT-3**.

---

### Core Pipeline Stages:
1. **Subword Tokenization with Byte Pair Encoding (BPE)**: Using OpenAI's `tiktoken` library to handle arbitrary out-of-vocabulary words without `<|unk|>` explosion.
2. **Data Sampling via Sliding Window**: Constructing input-target token pairs $(x, y)$ for autoregressive next-token prediction.
3. **PyTorch `Dataset` & `DataLoader`**: Efficient batching, shuffling, and windowed sequence slicing (`GPTDatasetV1`).
4. **Token Embedding Layer**: Projecting discrete token IDs into dense continuous vector spaces (`torch.nn.Embedding`).
5. **Positional Embedding Layer**: Encoding token position/order in a sequence.
6. **Final Input Embeddings**: Combining token and positional representations into unified input tensors ready for the Transformer architecture.

---
## 1. Loading Corpus Text
We load the raw text dataset that will be processed into training sequences.

In [115]:
with open("/home/lakshay/BuildingLargeLanguageModel/src/buildinglargelanguagemodel/data/the-verdict.txt", "r") as file:
    dataset = file.read()
    print(dataset[:100])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


---
## 2. Byte Pair Encoding (BPE) with `tiktoken`

### Why Byte Pair Encoding (BPE)?
- Word-level tokenizers fail on words absent from the training vocabulary (requiring `<|unk|>`).
- Character-level tokenizers preserve all words but result in excessively long sequence lengths, losing semantic abstractions.
- **BPE strikes the optimal balance**: It iteratively merges frequent character pairs into subwords. Unseen or rare words are gracefully broken down into smaller recognizable subword chunks or individual bytes, eliminating `<|unk|>` completely.

GPT-2 uses a BPE vocabulary of **50,257** tokens.

In [116]:
import tiktoken 

In [117]:
tokenizer = tiktoken.get_encoding("gpt2")

Test basic subword encoding and decoding on sample text:

In [118]:
tokenizer.encode("hey")

[20342]

In [119]:
tokenizer.decode(tokenizer.encode("hey"))

'hey'

### Tokenizing the Full Dataset
Let's encode our complete dataset into BPE token IDs and check the resulting token count:

In [ ]:
enc_text = tokenizer.encode(dataset)
enc_text

In [121]:
len(enc_text)

5145

Verify that decoding reproduces the original text losslessly:

In [ ]:
tokenizer.decode(enc_text)

### Handling Special Tokens in `tiktoken`
By default, `tiktoken` disables special tokens like `<|endoftext|>` to prevent prompt injection or unintended token splitting. To allow them, pass `allowed_special={"<|endoftext|>"}`:

In [123]:
text = "This is the specila <|endoftext|> token"

special_encode = tokenizer.encode(text, allowed_special= {"<|endoftext|>"})
special_encode

[1212, 318, 262, 1020, 10102, 220, 50256, 11241]

In [124]:
tokenizer.decode(special_encode)

'This is the specila <|endoftext|> token'

---
## 3. Data Sampling with a Sliding Window

In autoregressive language modeling (causal modeling), the task is to predict the **next token** given the preceding sequence:
- **Input ($x$)**: Sequence of $k$ tokens $[t_0, t_1, \dots, t_{k-1}]$
- **Target ($y$)**: Sequence shifted by 1 token to the right $[t_1, t_2, \dots, t_k]$

In [125]:
enc_sample = enc_text[50:]

In [126]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

print(f"Input x: {x}")
print(f"Taget y:       {y}")

Input x: [290, 4920, 2241, 287]
Taget y:       [4920, 2241, 287, 257]


---
## 4. Building the PyTorch Dataset & DataLoader

To train neural networks effectively, data must be batched, tensorized, and efficiently streamed:
- **`GPTDatasetV1`**: Slices token sequences using a sliding window parameterized by `max_length` (context size) and `stride` (step size).
- **`create_dataloader_v1`**: Wraps the dataset into a PyTorch `DataLoader` with batching, shuffling, and worker management.

In [127]:
import torch 

torch.__version__

'2.15.0+cu132'

In [128]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt,allowed_special={"<|endoftext|>"})            
            
        for i in range(0, len(token_ids) - max_length, stride):

            input_chunk = token_ids[i : i + max_length]

            target_chunk = token_ids[i + 1 : i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

Helper function to construct our batch data loader:

In [129]:
def create_dataloader_v1(txt, batch_size = 4, max_length = 256, stride = 128, shuffle = True, drop_last = True, num_workers = 0):

    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(txt , tokenizer ,max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle = shuffle,
        drop_last = drop_last,
        num_workers = num_workers,
    )

    return dataloader

### Inspecting a Batch of Training Pairs
Let's create a DataLoader with `batch_size=8`, `max_length=4`, and `stride=4`:

In [130]:
dataloader = create_dataloader_v1(
    dataset ,batch_size = 8, max_length = 4, stride = 4, shuffle = False
)

data_iter = iter(dataloader)
inputs ,targets = next(data_iter)
print("inputs : ", inputs)
print("targets : ", targets)

inputs :  tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
targets :  tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


Fetch the subsequent batch to observe how the sliding window steps forward:

In [131]:
data_iter = iter(dataloader)
inputs ,targets = next(data_iter)
print("inputs : ", inputs)
print("targets : ", targets)

inputs :  tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
targets :  tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


---
## 5. Token Embedding Layer Fundamentals

Before building full-scale embeddings, let's build intuition for `torch.nn.Embedding`:
- An embedding layer is essentially a lookup table of shape `[vocab_size, output_dim]`.
- For each integer token ID $i$, it retrieves the corresponding row vector $W[i]$.
- These vectors are trainable parameters that are updated via backpropagation.

In [132]:
input_ids = torch.tensor([2, 3 ,4 , 1 ,2 ])

Initialize a small toy embedding table: `vocab_size=6`, `output_dim=4`:

In [133]:
vocab_size = 6
output_dim = 4

torch.manual_seed(123)

embedding_layer  = torch.nn.Embedding(vocab_size, output_dim)

In [134]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880],
        [ 0.3486,  0.6603, -0.2196, -0.3792],
        [-0.1606, -0.4015,  0.6957, -1.8061],
        [ 1.8960, -0.1750,  1.3689, -1.6033],
        [-0.7849, -1.4096, -0.4076,  0.7953],
        [ 0.9985,  0.2212,  1.8319, -0.3378]], requires_grad=True)


Lookup a single token ID vs. a 1D tensor of multiple token IDs:

In [135]:
embedding_layer(torch.tensor([3]))

tensor([[ 1.8960, -0.1750,  1.3689, -1.6033]], grad_fn=<EmbeddingBackward0>)

In [136]:
embedding_layer(input_ids)

tensor([[-0.1606, -0.4015,  0.6957, -1.8061],
        [ 1.8960, -0.1750,  1.3689, -1.6033],
        [-0.7849, -1.4096, -0.4076,  0.7953],
        [ 0.3486,  0.6603, -0.2196, -0.3792],
        [-0.1606, -0.4015,  0.6957, -1.8061]], grad_fn=<EmbeddingBackward0>)

In [137]:
input_ids

tensor([2, 3, 4, 1, 2])

---
## 6. Scaling to GPT-2 Token Embeddings

Now we instantiate the real token embedding layer for GPT-2:
- `vocab_size = 50,257`
- `output_dim = 256` (reduced from GPT-2 small's 768 for demonstration)
- Inputs batch shape: `[batch_size, context_length]` = `[8, 4]`
- Output tensor shape: `[batch_size, context_length, output_dim]` = `[8, 4, 256]`

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size , output_dim)

In [139]:
max_length = 4

dataloader = create_dataloader_v1(
    dataset ,batch_size = 8, max_length = max_length, stride = 4, shuffle = False
)

data_iter = iter(dataloader)
inputs ,targets = next(data_iter)
print("inputs : ", inputs)
print("targets : ", targets)

inputs :  tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
targets :  tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [140]:
token_embeddings = token_embedding_layer(inputs)
token_embeddings.shape

torch.Size([8, 4, 256])

---
## 7. Positional Embeddings

### Why do we need Positional Embeddings?
The self-attention mechanism in Transformers is **permutation-equivariant**; it treats sequences as unordered bags of tokens. To provide the model with a sense of token order/position, we inject positional embeddings.

In GPT models, positions are represented using learned absolute positional embeddings:
- `pos_embedding_layer`: Embedding table of size `[context_length, output_dim]`.
- Input indices: `torch.arange(context_length)` $\rightarrow [0, 1, 2, \dots, \text{max\_length} - 1]$.

In [141]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length , output_dim)

Inspect raw position indices:

In [142]:
torch.arange(max_length)

tensor([0, 1, 2, 3])

Embed the position indices into `[context_length, output_dim]` vector space:

In [143]:
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)


torch.Size([4, 256])


---
## 8. Combining Token and Positional Embeddings

The final input representation fed into the Transformer blocks is the element-wise sum of:
$$\mathbf{X}_{\text{input}} = \mathbf{X}_{\text{token}} + \mathbf{X}_{\text{pos}}$$

PyTorch automatically broadcasts `pos_embeddings` (shape `[4, 256]`) across the batch dimension (`8`), producing a final input tensor of shape `[8, 4, 256]`.

In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


### Numerical Inspection of the Vector Addition
Compare the first token vector, its corresponding positional vector, and their final sum:

In [148]:
token_embeddings[0] 

tensor([[ 0.6802, -0.8658,  0.3554,  ...,  0.2981, -1.0365, -0.8160],
        [-0.7488,  0.1531,  1.0750,  ...,  1.8244,  0.4382, -0.3659],
        [ 0.8363, -1.5682, -1.5540,  ...,  1.2096, -0.3188, -0.5758],
        [-0.2773,  0.1254,  1.7587,  ...,  1.2567, -0.2673, -2.6502]],
       grad_fn=<SelectBackward0>)

In [149]:
pos_embeddings[0]

tensor([ 1.8998e+00, -1.4937e-01, -2.0049e+00,  5.8745e-01, -4.3538e-01,
        -1.0029e+00, -6.4243e-01, -1.3461e-01,  3.4127e-02,  2.1738e+00,
         1.1249e+00, -6.7244e-01, -1.3480e+00, -3.2319e-01,  6.7414e-01,
        -2.4317e+00, -4.2123e-01,  6.3503e-01, -2.0597e+00,  1.0767e-02,
        -8.9888e-01, -2.3412e+00, -7.9869e-01,  6.2235e-01, -1.7075e+00,
         9.2959e-01,  4.5603e-01,  8.6819e-01,  3.7480e-02,  9.3296e-01,
        -3.6707e-01, -2.5523e+00, -5.3225e-01,  7.9329e-01,  9.0319e-01,
        -2.1367e+00, -5.9144e-01, -2.6117e+00,  1.7093e-01,  1.6872e-01,
        -1.1584e+00,  1.3212e+00, -1.4642e-01, -1.0013e-01, -9.1768e-01,
         3.3704e-01,  5.8371e-01,  6.2518e-01,  1.0273e-02, -4.2007e-02,
        -1.1102e+00,  2.5731e+00,  1.5007e-01, -7.0643e-01, -2.3129e-01,
        -1.4990e-01, -1.1593e+00, -7.8594e-01, -1.0209e-01,  1.1111e+00,
         7.1180e-01,  1.3658e+00, -1.3581e+00,  5.7766e-01, -4.1971e-01,
         9.6131e-01,  1.0688e+00,  1.3325e+00, -1.3

In [150]:
input_embeddings[0]

tensor([[ 2.5800, -1.0151, -1.6495,  ...,  1.3364, -2.3303, -0.0233],
        [-2.8841, -0.4115,  1.3594,  ..., -0.5634, -0.0657,  0.3737],
        [ 0.8973, -1.8342, -0.6143,  ...,  1.5883,  1.3747, -0.1110],
        [-1.5777, -0.5812,  2.6580,  ...,  3.7430,  0.9456, -2.6424]],
       grad_fn=<SelectBackward0>)